# PyTorch GPU Usage & Distributed Training

Covers the full stack from basic device management through memory optimization, mixed precision, profiling, DataParallel, DistributedDataParallel, model parallelism, and gradient accumulation.

> **Note:** Cells that require multiple GPUs are clearly marked. Single-GPU cells run on any CUDA machine; CPU fallbacks are provided where meaningful.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms, models
import time
import os
import math
import numpy as np
import matplotlib.pyplot as plt

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU count       : {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name}  |  {props.total_memory / 1e9:.1f} GB  |  {props.multi_processor_count} SMs')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'\nDefault device  : {device}')

---
## 1. Device Management and Tensor Placement

The golden rule: **data and model must be on the same device**. PyTorch will raise a `RuntimeError` on device mismatch — it never silently moves tensors.

In [ ]:
# Creating tensors directly on a device
x_gpu = torch.randn(3, 4, device=device)
print(f'Tensor device: {x_gpu.device}')   # cuda:0 or cpu

# Moving existing tensors
x_cpu = torch.randn(3, 4)
x_gpu = x_cpu.to(device)          # preferred — device-agnostic
x_gpu = x_cpu.cuda()              # explicit CUDA (fails if no GPU)
x_cpu_back = x_gpu.cpu()          # back to CPU

# .to() is device-agnostic: if device='cpu', .to('cpu') is a no-op
print(f'Original:  {x_cpu.device}')
print(f'Moved:     {x_gpu.device}')
print(f'Back:      {x_cpu_back.device}')

In [ ]:
# Moving an entire model to a device
model = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
model = model.to(device)

# All parameters are now on the device
for name, param in model.named_parameters():
    print(f'{name:25s}  device={param.device}')

In [ ]:
# Device-agnostic code pattern — works on CPU and GPU without changes
def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# In a training loop:
def train_step(model, X, y, optimizer, criterion):
    X, y = X.to(device), y.to(device)   # move data each batch
    optimizer.zero_grad()
    loss = criterion(model(X), y)
    loss.backward()
    optimizer.step()
    return loss.item()

---
## 2. GPU Memory Management

Understanding GPU memory is essential for avoiding out-of-memory (OOM) errors and maximizing batch size.

In [ ]:
def mem_stats(label=''):
    if not torch.cuda.is_available(): return
    alloc   = torch.cuda.memory_allocated()  / 1e6
    reserved = torch.cuda.memory_reserved() / 1e6
    print(f'{label:30s}  allocated={alloc:.1f} MB  reserved={reserved:.1f} MB')

torch.cuda.empty_cache()  # release cached but unused memory back to OS
mem_stats('after empty_cache')

# Allocate a big tensor
big = torch.randn(1000, 1000, device=device)
mem_stats('after 1000x1000 alloc')

# Delete the Python reference
del big
mem_stats('after del (still reserved)')

# Force release of reserved cache
torch.cuda.empty_cache()
mem_stats('after empty_cache again')

In [ ]:
# Memory breakdown: parameters vs activations vs gradients vs optimizer state
def estimate_memory(model, batch_size, seq_or_spatial, dtype=torch.float32):
    """
    Rough estimate of GPU memory required for training.
    For a typical optimizer (Adam): 3x params (params + grad + 2 moment vectors)
    Activations depend on architecture and batch size.
    """
    bytes_per_element = torch.finfo(dtype).bits // 8
    n_params = sum(p.numel() for p in model.parameters())

    param_mem  = n_params * bytes_per_element
    grad_mem   = n_params * bytes_per_element
    adam_mem   = n_params * bytes_per_element * 2  # m and v

    print(f'Model parameters : {n_params:>12,}  ({param_mem/1e6:.1f} MB)')
    print(f'Gradients        : {n_params:>12,}  ({grad_mem/1e6:.1f} MB)')
    print(f'Adam moments     : {n_params*2:>12,}  ({adam_mem/1e6:.1f} MB)')
    print(f'Total (no acts)  :                  ({(param_mem+grad_mem+adam_mem)/1e6:.1f} MB)')
    print(f'Note: activations scale with batch_size={batch_size} and are architecture-dependent.')

resnet = models.resnet50()
estimate_memory(resnet, batch_size=32, seq_or_spatial=224)

In [ ]:
# Find maximum batch size that fits in memory via binary search
def max_batch_size(model, input_shape, device, lo=1, hi=512):
    """Binary search for max batch size. model must already be on device."""
    model.eval()
    last_good = lo
    while lo <= hi:
        mid = (lo + hi) // 2
        try:
            torch.cuda.empty_cache()
            x = torch.randn(mid, *input_shape, device=device)
            with torch.no_grad():
                _ = model(x)
            last_good = mid
            lo = mid + 1
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                hi = mid - 1
            else:
                raise
    return last_good

if torch.cuda.is_available():
    small_model = nn.Sequential(
        nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        nn.Linear(64, 10)
    ).to(device)
    bs = max_batch_size(small_model, (3, 224, 224), device, lo=1, hi=1024)
    print(f'Max batch size for model: {bs}')
else:
    print('Skipped: no GPU available')

---
## 3. Pinned Memory and Async Data Transfer

**Pinned (page-locked) memory** on the CPU allows the GPU DMA engine to transfer data directly, bypassing the OS and enabling **asynchronous** CPU→GPU transfers that overlap with GPU compute.

In [ ]:
def benchmark_transfer(size_mb=100, n_repeats=50):
    n_elements = int(size_mb * 1e6 / 4)  # float32

    # Pageable memory
    pageable = torch.randn(n_elements)
    t0 = time.perf_counter()
    for _ in range(n_repeats):
        _ = pageable.to(device, non_blocking=False)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t_pageable = (time.perf_counter() - t0) / n_repeats * 1000

    # Pinned memory
    pinned = torch.randn(n_elements).pin_memory()
    t0 = time.perf_counter()
    for _ in range(n_repeats):
        _ = pinned.to(device, non_blocking=True)   # async!
    if torch.cuda.is_available(): torch.cuda.synchronize()  # wait for all transfers
    t_pinned = (time.perf_counter() - t0) / n_repeats * 1000

    print(f'Transfer {size_mb} MB | pageable: {t_pageable:.2f} ms | pinned+async: {t_pinned:.2f} ms')
    if t_pageable > 0:
        print(f'Speedup: {t_pageable / t_pinned:.2f}x')

benchmark_transfer(100)

In [ ]:
# DataLoader settings for maximum throughput
# pin_memory=True: workers pre-allocate pinned memory, enabling async GPU transfer
# num_workers>0: background workers load/preprocess data while GPU computes
# persistent_workers: keep worker processes alive between epochs
# prefetch_factor: number of batches each worker prefetches ahead

dataset = datasets.FakeData(size=5000, image_size=(3, 224, 224), num_classes=10,
                             transform=transforms.ToTensor())

configs = [
    dict(num_workers=0,  pin_memory=False, label='no workers, no pin'),
    dict(num_workers=2,  pin_memory=False, label='2 workers, no pin'),
    dict(num_workers=2,  pin_memory=True,  label='2 workers + pin'),
    dict(num_workers=4,  pin_memory=True,  label='4 workers + pin'),
]

for cfg in configs:
    label = cfg.pop('label')
    loader = DataLoader(dataset, batch_size=64, **cfg)
    t0 = time.perf_counter()
    for X, y in loader:
        X = X.to(device, non_blocking=cfg.get('pin_memory', False))
    elapsed = time.perf_counter() - t0
    print(f'{label:30s}  {elapsed:.2f}s')

---
## 4. Mixed Precision Training (AMP)

**Automatic Mixed Precision** runs the forward pass in `float16` (half precision) where safe, keeping `float32` for numerically sensitive operations (softmax, loss, batch norm). Benefits:

- ~2× memory reduction (activations stored as fp16)
- 2–3× compute speedup on Tensor Core GPUs (V100, A100, RTX series)
- Requires a **GradScaler** to prevent fp16 gradient underflow

In [ ]:
# Why fp16 needs a GradScaler:
# fp16 range: ~6e-5 to 65504. Small gradients (e.g. 1e-7) underflow to 0.
# Solution: multiply loss by a large scale factor S before backward,
# then divide gradients by S before the optimizer step.
# The scaler automatically adjusts S up/down based on overflow detection.

if torch.cuda.is_available():
    from torch.cuda.amp import autocast, GradScaler

    model_amp = models.resnet18(num_classes=10).to(device)
    optimizer_amp = optim.AdamW(model_amp.parameters(), lr=1e-3)
    scaler = GradScaler()
    criterion = nn.CrossEntropyLoss()

    X_fake = torch.randn(32, 3, 224, 224, device=device)
    y_fake = torch.randint(0, 10, (32,), device=device)

    # --- fp32 baseline ---
    t0 = time.perf_counter()
    for _ in range(10):
        optimizer_amp.zero_grad()
        loss = criterion(model_amp(X_fake), y_fake)
        loss.backward()
        optimizer_amp.step()
    torch.cuda.synchronize()
    t_fp32 = (time.perf_counter() - t0) / 10 * 1000

    # --- fp16 with AMP ---
    t0 = time.perf_counter()
    for _ in range(10):
        optimizer_amp.zero_grad()
        with autocast():                          # fp16 forward
            loss = criterion(model_amp(X_fake), y_fake)
        scaler.scale(loss).backward()             # scaled backward
        scaler.step(optimizer_amp)                # unscale + optimizer step
        scaler.update()                           # adjust scale factor
    torch.cuda.synchronize()
    t_amp = (time.perf_counter() - t0) / 10 * 1000

    print(f'fp32 step: {t_fp32:.1f} ms')
    print(f'AMP  step: {t_amp:.1f} ms')
    print(f'Speedup:   {t_fp32/t_amp:.2f}x')
else:
    print('Skipped: requires CUDA')

In [ ]:
# Full AMP training loop template
def train_amp(model, loader, optimizer, criterion, n_epochs=3):
    scaler = GradScaler(enabled=torch.cuda.is_available())
    model.train()
    for epoch in range(n_epochs):
        for X, y in loader:
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(grad_set_to_none=True)  # faster than zero_grad()
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(X)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            # Optionally clip gradients (must unscale first)
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        print(f'Epoch {epoch+1} | scale={scaler.get_scale():.0f} | loss={loss.item():.4f}')

---
## 5. Gradient Accumulation

Simulates a larger effective batch size when GPU memory cannot fit the desired batch. Accumulate gradients over $K$ mini-batches before taking an optimizer step — mathematically equivalent to one step with a batch $K$ times larger.

In [ ]:
# Effective batch size = physical_batch_size * accumulation_steps
# e.g. 32 samples * 8 steps = 256 effective batch size

def train_with_accumulation(model, loader, optimizer, criterion,
                             accumulation_steps=8, n_epochs=3):
    scaler = GradScaler(enabled=torch.cuda.is_available())
    model.train()
    for epoch in range(n_epochs):
        optimizer.zero_grad(grad_set_to_none=True)
        for step, (X, y) in enumerate(loader):
            X, y = X.to(device), y.to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(X)
                # Divide loss by accumulation steps so gradients are correctly scaled
                loss = criterion(logits, y) / accumulation_steps
            scaler.scale(loss).backward()

            if (step + 1) % accumulation_steps == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(grad_set_to_none=True)

        print(f'Epoch {epoch+1} done')

---
## 6. PyTorch Profiler

The PyTorch profiler traces GPU kernel execution, CPU ops, memory allocation, and identifies bottlenecks.

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

model_prof = models.resnet18(num_classes=10).to(device)
X_prof = torch.randn(16, 3, 224, 224, device=device)
y_prof = torch.randint(0, 10, (16,), device=device)
optimizer_prof = optim.SGD(model_prof.parameters(), lr=0.01)

activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    record_shapes=True,
    profile_memory=True,
    with_stack=False,
) as prof:
    for _ in range(5):
        with record_function('forward'):
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model_prof(X_prof)
                loss = F.cross_entropy(logits, y_prof)
        with record_function('backward'):
            loss.backward()
        with record_function('optimizer'):
            optimizer_prof.step()
            optimizer_prof.zero_grad(grad_set_to_none=True)

print(prof.key_averages().table(sort_by='cuda_time_total' if torch.cuda.is_available()
                                 else 'cpu_time_total', row_limit=15))

In [ ]:
# Export a Chrome trace viewable at chrome://tracing
# prof.export_chrome_trace('trace.json')

# Filter to the top memory-consuming ops
print(prof.key_averages().table(sort_by='self_cpu_memory_usage', row_limit=10))

---
## 7. `torch.compile` — PyTorch 2.x Graph Compilation

`torch.compile` traces the model into a computation graph and compiles it via TorchInductor (generates Triton kernels on GPU). Typically 1.5–3× speedup with no code changes.

In [ ]:
if torch.__version__ >= '2.0':
    model_base    = models.resnet18(num_classes=10).to(device).eval()
    model_compiled = torch.compile(model_base, mode='reduce-overhead')  # or 'max-autotune'

    X_bench = torch.randn(32, 3, 224, 224, device=device)

    def benchmark(fn, n_warmup=5, n_bench=20):
        for _ in range(n_warmup):
            fn()
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_bench):
            fn()
        if torch.cuda.is_available(): torch.cuda.synchronize()
        return (time.perf_counter() - t0) / n_bench * 1000

    with torch.no_grad():
        t_base     = benchmark(lambda: model_base(X_bench))
        t_compiled = benchmark(lambda: model_compiled(X_bench))

    print(f'Eager:    {t_base:.2f} ms')
    print(f'Compiled: {t_compiled:.2f} ms')
    print(f'Speedup:  {t_base/t_compiled:.2f}x')
else:
    print('torch.compile requires PyTorch >= 2.0')

---
## 8. `nn.DataParallel` — Single-Machine Multi-GPU (Simple)

`DataParallel` splits each batch across GPUs, runs forward in parallel, gathers outputs to GPU 0 for the loss, then scatters gradients back. Simple to use but has significant overhead:

- **GPU 0 bottleneck:** loss and parameter updates happen only on GPU 0
- **GIL:** Python's GIL limits parallelism at the Python level
- **Imbalanced memory:** GPU 0 holds a full copy plus gathered activations
- **Not recommended** for serious multi-GPU work — use DDP instead

Useful for quick experiments when you have 2–4 GPUs and don't want to restructure code.

In [ ]:
n_gpus = torch.cuda.device_count()
print(f'Available GPUs: {n_gpus}')

if n_gpus >= 2:
    model_dp = models.resnet18(num_classes=10)
    model_dp = nn.DataParallel(model_dp, device_ids=[0, 1])  # use GPUs 0 and 1
    model_dp = model_dp.to('cuda:0')  # primary device
    print(f'DataParallel wraps: {model_dp.module.__class__.__name__}')
    # Access underlying model: model_dp.module
    X = torch.randn(64, 3, 224, 224).to('cuda:0')
    out = model_dp(X)
    print(f'Output shape: {out.shape}')  # batch split across GPUs, gathered on cuda:0
else:
    print('DataParallel example requires >= 2 GPUs')

---
## 9. `DistributedDataParallel` (DDP) — The Right Way

DDP launches one **process per GPU**. Each process has its own model replica. Gradients are synchronized via **AllReduce** (ring-allreduce) after the backward pass — no single-GPU bottleneck.

### Key concepts

| Concept | Meaning |
|---|---|
| `world_size` | Total number of processes (= GPUs for single-node) |
| `rank` | Unique ID of this process (0 to world_size-1) |
| `local_rank` | GPU index on this node |
| `backend` | Communication library: `nccl` (GPU), `gloo` (CPU/GPU), `mpi` |
| AllReduce | Averages gradients across all ranks simultaneously — $O(\log N)$ steps |

### Why AllReduce beats DataParallel's gather

DataParallel gathers all gradients to GPU 0, which is $O(N)$ data movement on one link. Ring-allreduce sends $O(2(N-1)/N)$ data per GPU — balanced and faster at scale.

In [ ]:
# DDP requires spawning separate processes — it cannot run interactively in a notebook cell.
# Below is the complete DDP training script that you would save as train_ddp.py and launch with:
#
#   torchrun --nproc_per_node=NUM_GPUS train_ddp.py
#
# or (older):
#   python -m torch.distributed.launch --nproc_per_node=NUM_GPUS train_ddp.py

DDP_SCRIPT = '''
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler
from torchvision import datasets, transforms, models

# --------------------------------------------------------------------------
# 1. Initialise the process group
# --------------------------------------------------------------------------
def setup():
    # torchrun sets RANK, LOCAL_RANK, WORLD_SIZE automatically
    dist.init_process_group(backend="nccl")  # nccl is fastest for GPU-GPU
    torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))

def cleanup():
    dist.destroy_process_group()

# --------------------------------------------------------------------------
# 2. Model: wrap in DDP after moving to the correct device
# --------------------------------------------------------------------------
def build_model():
    local_rank = int(os.environ["LOCAL_RANK"])
    device = torch.device(f"cuda:{local_rank}")
    model = models.resnet18(num_classes=10).to(device)
    # DDP synchronises batch-norm stats across processes too
    model = DDP(model, device_ids=[local_rank], output_device=local_rank)
    return model, device

# --------------------------------------------------------------------------
# 3. Data: DistributedSampler ensures each rank sees a different subset
# --------------------------------------------------------------------------
def build_loaders():
    transform = transforms.Compose([transforms.ToTensor(),
                                     transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
    dataset = datasets.CIFAR10("./data", train=True, download=True, transform=transform)
    # DistributedSampler shards the dataset across ranks without overlap
    sampler = DistributedSampler(dataset, shuffle=True)
    loader = DataLoader(dataset, batch_size=64, sampler=sampler,
                        num_workers=4, pin_memory=True)
    return loader, sampler

# --------------------------------------------------------------------------
# 4. Training loop
# --------------------------------------------------------------------------
def train(n_epochs=5):
    setup()
    rank = dist.get_rank()
    model, device = build_model()
    loader, sampler = build_loaders()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.cuda.amp.GradScaler()
    criterion = nn.CrossEntropyLoss()

    for epoch in range(n_epochs):
        sampler.set_epoch(epoch)  # ensures different shuffle each epoch
        model.train()
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad(grad_set_to_none=True)
            with torch.cuda.amp.autocast():
                loss = criterion(model(X), y)
            scaler.scale(loss).backward()
            # Gradients are automatically allreduced by DDP before this step
            scaler.step(optimizer)
            scaler.update()

        # Only rank 0 should log or save checkpoints
        if rank == 0:
            print(f"Epoch {epoch+1} | loss={loss.item():.4f}")
            torch.save(model.module.state_dict(), f"checkpoint_epoch{epoch+1}.pt")
            # model.module accesses the underlying unwrapped model

    cleanup()

if __name__ == "__main__":
    train()
'''

print('DDP script content:')
print(DDP_SCRIPT)

In [ ]:
# Save the DDP script to disk so you can run it
with open('train_ddp.py', 'w') as f:
    f.write(DDP_SCRIPT)
print('Saved to train_ddp.py')
print()
print('Launch command:')
print('  torchrun --nproc_per_node=2 train_ddp.py   # 2-GPU single node')
print('  torchrun --nproc_per_node=8 train_ddp.py   # 8-GPU single node')
print()
print('Multi-node (2 nodes, 8 GPUs each):')
print('  # On node 0 (master):')
print('  torchrun --nnodes=2 --nproc_per_node=8 --node_rank=0 \\')
print('           --master_addr=<node0_ip> --master_port=29500 train_ddp.py')
print('  # On node 1:')
print('  torchrun --nnodes=2 --nproc_per_node=8 --node_rank=1 \\')
print('           --master_addr=<node0_ip> --master_port=29500 train_ddp.py')

---
## 10. DDP Communication Primitives

Sometimes you need manual collective operations — e.g., aggregating metrics across ranks after evaluation.

In [ ]:
# These require an initialised process group; shown here for reference

DIST_PRIMITIVES = '''
import torch.distributed as dist

# --- AllReduce: sum (or avg) a tensor across all ranks ---
tensor = torch.tensor(rank_specific_value, device=device)
dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
tensor /= dist.get_world_size()   # average

# --- Broadcast: send tensor from rank 0 to all others ---
dist.broadcast(tensor, src=0)

# --- AllGather: each rank contributes its tensor, all ranks receive all ---
gather_list = [torch.zeros_like(tensor) for _ in range(world_size)]
dist.all_gather(gather_list, tensor)

# --- Reduce: like AllReduce but only rank `dst` receives the result ---
dist.reduce(tensor, dst=0, op=dist.ReduceOp.SUM)

# --- Barrier: block until all ranks reach this point ---
dist.barrier()

# Common pattern: aggregate val accuracy across ranks after eval
# Each rank computes (correct, total) on its shard
correct_t = torch.tensor([correct], dtype=torch.float32, device=device)
total_t   = torch.tensor([total],   dtype=torch.float32, device=device)
dist.all_reduce(correct_t, op=dist.ReduceOp.SUM)
dist.all_reduce(total_t,   op=dist.ReduceOp.SUM)
if rank == 0:
    global_acc = correct_t.item() / total_t.item()
    print(f"Global val accuracy: {global_acc:.4f}")
'''
print(DIST_PRIMITIVES)

---
## 11. Model Parallelism

When a single model is too large to fit on one GPU, split it across devices. Different layers run on different GPUs; a batch flows sequentially through them.

**Pipeline parallelism** (GPipe, PipelineParallel) improves utilization by splitting the batch into micro-batches so multiple GPUs can be active simultaneously.

In [ ]:
class ModelParallelResNet(nn.Module):
    """
    Splits ResNet across two GPUs.
    Requires at least 2 GPUs — shown here as a structural example.
    If only 1 GPU is available, both halves run on the same device.
    """
    def __init__(self):
        super().__init__()
        self.dev0 = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        self.dev1 = torch.device('cuda:1' if torch.cuda.device_count() > 1 else self.dev0)

        base = models.resnet50()
        # First half: layers 0..5 on GPU 0
        self.part1 = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2
        ).to(self.dev0)
        # Second half: layers 6..end on GPU 1
        self.part2 = nn.Sequential(
            base.layer3, base.layer4,
            base.avgpool
        ).to(self.dev1)
        self.fc = base.fc.to(self.dev1)

    def forward(self, x):
        x = self.part1(x.to(self.dev0))          # compute on GPU 0
        x = self.part2(x.to(self.dev1))          # move to GPU 1, compute
        return self.fc(x.view(x.size(0), -1))


mp_model = ModelParallelResNet()
print('Model parallel ResNet created')
print(f'  part1 device: {next(mp_model.part1.parameters()).device}')
print(f'  part2 device: {next(mp_model.part2.parameters()).device}')

# The .to(self.dev1) in forward is the inter-GPU tensor copy — this has latency.
# Minimise it by keeping copies as infrequent as possible.

In [ ]:
# Pipeline parallelism concept — micro-batching hides the inter-GPU transfer latency

PIPELINE_CONCEPT = '''
# Without pipelining (bubble problem):
# Time: |--GPU0 forward--|--GPU1 forward--|--GPU1 backward--|--GPU0 backward--|
#         GPU1 idle ^^                     GPU0 idle ^^^

# With micro-batching (GPipe):
# Split batch into M micro-batches. GPU 0 processes micro-batch 2 while
# GPU 1 processes micro-batch 1 — reducing idle "bubble" time.
#
# Bubble fraction = (num_stages - 1) / (num_stages - 1 + num_micro_batches)
# -> more micro-batches = smaller bubble

# PyTorch native pipeline parallel:
from torch.distributed.pipeline.sync import Pipe
# model must be nn.Sequential, each stage on a different device
pipeline_model = Pipe(sequential_model, chunks=8)  # 8 micro-batches
'''
print(PIPELINE_CONCEPT)

---
## 12. Tensor Parallelism and FSDP

For very large models (billions of parameters) that don't fit across even many GPUs with model parallelism alone.

In [ ]:
FSDP_OVERVIEW = '''
# Fully Sharded Data Parallel (FSDP) — PyTorch native (torch.distributed.fsdp)
#
# DDP: each rank holds a full copy of the model. Memory = model_size * world_size.
# FSDP: each rank holds only 1/world_size of the model parameters, gradients,
#       and optimizer states. Gathers full layer parameters only when needed
#       (forward/backward through that layer), then discards them.
#
# Memory: ~model_size / world_size (near linear scaling with GPU count)
#
# This enables training models larger than a single GPU's VRAM.

from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
import functools

# Auto-wrap: shard layers with more than 1M parameters
auto_wrap = functools.partial(size_based_auto_wrap_policy, min_num_params=1_000_000)

model = FSDP(
    model,
    auto_wrap_policy=auto_wrap,
    mixed_precision=MixedPrecision(
        param_dtype=torch.float16,
        reduce_dtype=torch.float32,   # accumulate gradients in fp32
        buffer_dtype=torch.float16,
    ),
    sharding_strategy=ShardingStrategy.FULL_SHARD,  # or HYBRID_SHARD for multi-node
    device_id=torch.cuda.current_device(),
)

# Saving checkpoints with FSDP requires gathering all shards:
with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT):
    state = model.state_dict()  # only rank 0 will have the full state dict
    if dist.get_rank() == 0:
        torch.save(state, "model.pt")
'''
print(FSDP_OVERVIEW)

---
## 13. Checkpointing and Resuming

Proper checkpoint discipline is critical for long distributed runs.

In [ ]:
def save_checkpoint(model, optimizer, scheduler, scaler, epoch, loss, path):
    """
    Save everything needed to resume training exactly.
    For DDP: use model.module.state_dict() to strip the DDP wrapper.
    """
    checkpoint = {
        'epoch'           : epoch,
        'model_state'     : model.state_dict(),     # or model.module.state_dict() for DDP
        'optimizer_state' : optimizer.state_dict(),
        'scheduler_state' : scheduler.state_dict() if scheduler else None,
        'scaler_state'    : scaler.state_dict()    if scaler    else None,
        'loss'            : loss,
    }
    torch.save(checkpoint, path)
    print(f'Checkpoint saved to {path}')


def load_checkpoint(path, model, optimizer, scheduler=None, scaler=None, device='cpu'):
    """
    Load and restore all states. map_location handles moving a GPU checkpoint
    to a different device (important for DDP where each rank loads its own copy).
    """
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    if scheduler and ckpt['scheduler_state']:
        scheduler.load_state_dict(ckpt['scheduler_state'])
    if scaler and ckpt['scaler_state']:
        scaler.load_state_dict(ckpt['scaler_state'])
    print(f'Resumed from epoch {ckpt["epoch"]} | loss={ckpt["loss"]:.4f}')
    return ckpt['epoch'] + 1  # return next epoch to continue from


# Demo save/load
small_m   = nn.Linear(10, 2).to(device)
small_opt = optim.Adam(small_m.parameters())
scaler_ck  = GradScaler(enabled=torch.cuda.is_available()) if torch.cuda.is_available() else None

save_checkpoint(small_m, small_opt, None, scaler_ck, epoch=5, loss=0.42, path='demo_ckpt.pt')
next_epoch = load_checkpoint('demo_ckpt.pt', small_m, small_opt, device=device)
print(f'Will resume from epoch {next_epoch}')

---
## 14. GPU Utilization Benchmarks — CPU vs GPU Scaling

In [ ]:
def time_matmul(size, device, n=50):
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)
    if str(device) != 'cpu': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        _ = a @ b
    if str(device) != 'cpu': torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000  # ms

sizes = [64, 128, 256, 512, 1024, 2048, 4096]
cpu_times, gpu_times = [], []

for s in sizes:
    t_cpu = time_matmul(s, torch.device('cpu'), n=10)
    cpu_times.append(t_cpu)
    if torch.cuda.is_available():
        t_gpu = time_matmul(s, torch.device('cuda:0'), n=50)
        gpu_times.append(t_gpu)
    print(f'{s:5d}x{s:5d}  CPU: {t_cpu:8.2f} ms' +
          (f'  GPU: {gpu_times[-1]:8.3f} ms  speedup: {t_cpu/gpu_times[-1]:6.1f}x'
           if gpu_times else ''))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.loglog(sizes, cpu_times, 'o-', label='CPU')
if gpu_times: plt.loglog(sizes, gpu_times, 's-', label='GPU')
plt.xlabel('Matrix size'); plt.ylabel('Time (ms)'); plt.title('MatMul time'); plt.legend()
plt.subplot(1, 2, 2)
if gpu_times:
    speedups = [c/g for c, g in zip(cpu_times, gpu_times)]
    plt.semilogx(sizes, speedups, 'D-')
    plt.xlabel('Matrix size'); plt.ylabel('GPU speedup'); plt.title('CPU/GPU Speedup ratio')
plt.tight_layout(); plt.show()

---
## 15. Summary: Training Configuration Cheat Sheet

| Scenario | Recommended setup |
|---|---|
| Single GPU, small model | `model.to(device)`, fp32 |
| Single GPU, large model | AMP (`autocast` + `GradScaler`) + gradient accumulation |
| 2–4 GPUs, same machine | DDP with `torchrun --nproc_per_node=N` |
| 8+ GPUs, same machine | DDP + NCCL backend + AMP |
| Multi-node | DDP + `torchrun --nnodes=N` + NCCL |
| Model too big for 1 GPU | Model parallelism or FSDP |
| Billions of params | FSDP + tensor parallelism (e.g., Megatron-LM) |
| Quick experiment, 2 GPUs | `nn.DataParallel` (acceptable but suboptimal) |

### Memory optimization checklist

1. Use `grad_set_to_none=True` in `optimizer.zero_grad()` — avoids allocating a zero tensor
2. Enable AMP — halves activation memory
3. Use `pin_memory=True` + `non_blocking=True` — overlaps transfer with compute
4. Use `torch.no_grad()` during validation — no activation graph stored
5. Delete intermediate tensors and call `torch.cuda.empty_cache()` after OOM recovery
6. Use gradient checkpointing (`torch.utils.checkpoint`) — trades compute for memory by recomputing activations during backward
7. Increase batch size until GPU memory is ~80% utilized — low utilization means wasted hardware

### Gradient checkpointing
```python
from torch.utils.checkpoint import checkpoint

# Instead of:
out = expensive_layer(x)   # saves activations for backward

# Use:
out = checkpoint(expensive_layer, x)  # recomputes forward during backward
# Saves ~sqrt(L) memory for L layers at the cost of one extra forward pass
```